# Head-to-head: vstash vs ColBERTv2 (T4 Colab)

Same conditions, two retrieval engines, three benchmark families:

  1. **LongMemEval-s** (multi-session chat memory): per-question
     fresh in-memory index, session-level Recall@K.
  2. **BEIR** (5 datasets: SciFact, NFCorpus, FiQA, SciDocs,
     ArguAna): NDCG@10, MRR, Recall@10 against published qrels.

## Engine comparison

| | vstash | ColBERTv2 |
|---|---|---|
| Model | BAAI/bge-small-en-v1.5 (33M, 384d single-vec) | colbert-ir/colbertv2.0 (~110M, 128d per token) |
| Search | vector ANN + FTS5 + adaptive RRF + MMR | multi-vector MaxSim |
| Storage | sqlite-vec + FTS5 in one .db file | per-corpus in-memory tensor |
| Doc token cap | 512 | ~220 |

## Why this is a real H2H

The paper today cites **published** ColBERTv2 numbers from the
BEIR / ColBERTv2 papers.  That is fair on BEIR (same dataset and
metric) but says nothing about chat data.  This notebook *runs*
ColBERTv2 ourselves on identical inputs to vstash so:

  - LongMemEval is a brand new comparison (no published number).
  - BEIR adds a same-machine sanity check on top of the published
    numbers (any gap from environment / library drift surfaces).

Outputs are saved to Drive `MyDrive/lme_h2h/` immediately after each
stage so a runtime disconnect does not lose hours of work.

In [ ]:
# Cell 1: Setup -- vstash from the experiment branch + pylate.
BRANCH = 'feature/longmemeval-retrain-experiments'

!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0' pylate huggingface_hub
%cd /content
!rm -rf /content/vstash
!git clone --branch $BRANCH https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .

import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
print('cuda:', torch.cuda.get_device_name(0),
      '| mem GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# Cell 2: Mount Drive up front (disconnect-safe).
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/lme_h2h'
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs('/content/results', exist_ok=True)
print('Drive mounted, output dir:', DRIVE_OUT)

In [ ]:
# Cell 3: Download longmemeval_s for the chat-memory benchmark.
import os
from huggingface_hub import hf_hub_download
TARGET = '/content/vstash/experiments/data/longmemeval'
os.makedirs(TARGET, exist_ok=True)
p = hf_hub_download('xiaowu0162/longmemeval', 'longmemeval_s',
                    repo_type='dataset', local_dir=TARGET)
print(f'Downloaded ({os.path.getsize(p) / 1024 / 1024:.1f} MB) -> {p}')

## LongMemEval (chat memory)

In [ ]:
# Cell 4: vstash on LongMemEval-s (full 500 questions, hybrid mode).
# ~9 min on Colab CPU FastEmbed.
import os
os.chdir('/content/vstash')
!python -m experiments.longmemeval_retrieval \
    --all \
    --output /content/results/lme_full_500_vstash.json
!cp /content/results/lme_full_500_vstash.json $DRIVE_OUT/
print('Saved to Drive.')

In [ ]:
# Cell 5: ColBERTv2 on LongMemEval-s (full 500, per-question fresh index).
# ~30-45 min: encoding ~50 sessions per question dominates.
import os
os.chdir('/content/vstash')
!python -m experiments.longmemeval_colbert \
    --all \
    --device cuda \
    --encode-batch-size 32 \
    --output /content/results/lme_full_500_colbertv2.json
!cp /content/results/lme_full_500_colbertv2.json $DRIVE_OUT/
print('Saved to Drive.')

## BEIR (5 datasets, published qrels)

In [ ]:
# Cell 6: vstash on the 5 BEIR datasets used in the paper.  Reuses
# experiments.beir_benchmark with --no-chroma (matches our recommended
# default).  This regenerates the headline NDCG@10 row of the paper.
# ~10-20 min depending on FiQA wait.
import os
os.chdir('/content/vstash')
!python -m experiments.beir_benchmark --no-chroma \
    --datasets scifact nfcorpus fiqa scidocs arguana
# beir_benchmark writes its summary to experiments/results/beir_benchmark.json.
import shutil
src = '/content/vstash/experiments/results/beir_benchmark.json'
dst_local = '/content/results/beir_vstash.json'
if os.path.isfile(src):
    shutil.copy(src, dst_local)
    !cp $dst_local $DRIVE_OUT/
    print(f'Saved {dst_local} + Drive copy.')
else:
    print(f'WARN: {src} not found -- check the script output for the actual path.')

In [ ]:
# Cell 7: ColBERTv2 on the same 5 BEIR datasets.  Same metrics
# definitions (ndcg_at_k / mrr / recall_at_k from beir_benchmark.py),
# same qrels, same TOP_K=10.  ~15-25 min depending on FiQA encoding.
import os
os.chdir('/content/vstash')
!python -m experiments.beir_colbert \
    --datasets scifact nfcorpus fiqa scidocs arguana \
    --device cuda \
    --output /content/results/beir_colbertv2.json
!cp /content/results/beir_colbertv2.json $DRIVE_OUT/
print('Saved to Drive.')

## Comparison tables

In [ ]:
# Cell 8: Build side-by-side tables (LongMemEval + BEIR) and persist.
import json
import os

lme_v = json.load(open('/content/results/lme_full_500_vstash.json'))
lme_c = json.load(open('/content/results/lme_full_500_colbertv2.json'))
if os.path.isfile('/content/results/beir_vstash.json'):
    beir_v_raw = json.load(open('/content/results/beir_vstash.json'))
    beir_v = beir_v_raw.get('results', beir_v_raw) if isinstance(beir_v_raw, dict) else beir_v_raw
else:
    beir_v = []
beir_c = json.load(open('/content/results/beir_colbertv2.json')) if os.path.isfile('/content/results/beir_colbertv2.json') else []

ks = (1, 3, 5, 10, 20, 50)
lines: list[str] = []
lines.append('LongMemEval-s (Recall@K, session-level, n=500):')
lines.append(f'  {"K":>3} | {"vstash":>8} | {"ColBERT":>8} | {"delta":>8}')
lines.append('  ' + '-' * 38)
for k in ks:
    vk = lme_v['summary']['macro'][f'recall@{k}']
    ck = lme_c['summary']['macro'][f'recall@{k}']
    lines.append(f'  {k:>3} | {vk:>8.4f} | {ck:>8.4f} | {vk - ck:+8.4f}')

lines.append('')
lines.append('LongMemEval per question_type (Recall@10):')
for t in sorted(lme_v['summary']['by_question_type']):
    vk = lme_v['summary']['by_question_type'][t]['recall@10']
    ck = lme_c['summary']['by_question_type'][t]['recall@10']
    nq = lme_v['summary']['by_question_type'][t]['n']
    lines.append(
        f'  {t:30s} n={nq:3d} | vstash={vk:.4f} | ColBERT={ck:.4f} | delta={vk - ck:+.4f}'
    )

lines.append('')
lines.append('BEIR (NDCG@10, published qrels):')
lines.append(f'  {"Dataset":<10} | {"vstash":>7} | {"ColBERT":>7} | {"delta":>8}')
lines.append('  ' + '-' * 42)
by_ds_v = {r['dataset']: r for r in beir_v}
by_ds_c = {r['dataset']: r for r in beir_c}
for ds in ('scifact', 'nfcorpus', 'fiqa', 'scidocs', 'arguana'):
    rv = by_ds_v.get(ds, {}).get('vstash', {}).get('ndcg_10')
    rc = by_ds_c.get(ds, {}).get('colbertv2', {}).get('ndcg_10')
    if rv is None or rc is None:
        lines.append(f'  {ds:<10} | {"-":>7} | {"-":>7} | {"-":>8}')
    else:
        lines.append(f'  {ds:<10} | {rv:>7.4f} | {rc:>7.4f} | {rv - rc:+8.4f}')

lines.append('')
lines.append('BEIR (Recall@10):')
lines.append(f'  {"Dataset":<10} | {"vstash":>7} | {"ColBERT":>7} | {"delta":>8}')
lines.append('  ' + '-' * 42)
for ds in ('scifact', 'nfcorpus', 'fiqa', 'scidocs', 'arguana'):
    rv = by_ds_v.get(ds, {}).get('vstash', {}).get('recall_10')
    rc = by_ds_c.get(ds, {}).get('colbertv2', {}).get('recall_10')
    if rv is None or rc is None:
        lines.append(f'  {ds:<10} | {"-":>7} | {"-":>7} | {"-":>8}')
    else:
        lines.append(f'  {ds:<10} | {rv:>7.4f} | {rc:>7.4f} | {rv - rc:+8.4f}')

table = '\n'.join(lines)
print(table)

comparison = {
    'longmemeval': {
        'vstash': lme_v['summary'],
        'colbertv2': lme_c['summary'],
        'delta_macro': {
            f'recall@{k}': lme_v['summary']['macro'][f'recall@{k}']
            - lme_c['summary']['macro'][f'recall@{k}'] for k in ks
        },
    },
    'beir': {
        'vstash': by_ds_v,
        'colbertv2': by_ds_c,
    },
    'table': table,
}
out = '/content/results/h2h_full.json'
with open(out, 'w') as fh:
    json.dump(comparison, fh, indent=2)
!cp $out $DRIVE_OUT/
print(f'\nWrote {out} + Drive copy.')